# SEDI Index

In [ ]:
# general use packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import numpy as np

# packages for altering time to match up!
import sys
import cftime

# climpred packages
import climpred
from climpred import HindcastEnsemble
from climpred.tutorial import load_dataset
from climpred.stats import rm_poly

# SMYLE Utility functions
from SMYLEutils import io_utils as io
from SMYLEutils import calendar_utils as cal
from SMYLEutils import stat_utils as stat
import random

In [ ]:
var = 'totC_zint_100m' # photoC_tot_zint_100m , totChl_zint_100m , totC_zint_100m
var2 = 'totC_zint_100m' # photoC_tot_zint_100m , totChl_zint_100m , totC_zint_100m
depth = 'surface'
level= '0.9'

In [ ]:
smyle02 = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/extremes/' + var + '.monthly.2.regrid.extremes.' + str(level) + '.nc').extreme
smyle02_time = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/' + var2 + '.monthly.2.time.nc')

smyle05 = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/extremes/' + var + '.monthly.5.regrid.extremes.' + str(level) + '.nc').extreme
smyle05_time = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/' + var2 + '.monthly.5.time.nc')

smyle08 = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/extremes/' + var + '.monthly.8.regrid.extremes.' + str(level) + '.nc').extreme
smyle08_time = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/' + var2 + '.monthly.8.time.nc')

smyle11 = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/extremes/' + var + '.monthly.11.regrid.extremes.' + str(level) + '.nc').extreme
smyle11_time = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/' + var2 + '.monthly.11.time.nc')

In [ ]:
smyle02 = smyle02.expand_dims("init"); smyle02_time  = smyle02_time.expand_dims("init")
smyle05 = smyle05.expand_dims("init"); smyle05_time  = smyle05_time.expand_dims("init")
smyle08 = smyle08.expand_dims("init"); smyle08_time  = smyle08_time.expand_dims("init")
smyle11 = smyle11.expand_dims("init"); smyle11_time  = smyle11_time.expand_dims("init")

In [ ]:
smyle02['init'] = ['02']; smyle02_time['init'] = ['02']
smyle05['init'] = ['05']; smyle05_time['init'] = ['05']
smyle08['init'] = ['08']; smyle08_time['init'] = ['08']
smyle11['init'] = ['11']; smyle11_time['init'] = ['11']

In [ ]:
# subset!
smyle02 = smyle02.isel(L=slice(0,30)); smyle02_time = smyle02_time.isel(L=slice(0,30))
smyle05 = smyle05.isel(L=slice(0,30)); smyle05_time = smyle05_time.isel(L=slice(0,30))
smyle08 = smyle08.isel(L=slice(0,30)); smyle08_time = smyle08_time.isel(L=slice(0,30))
smyle11 = smyle11.isel(L=slice(0,30)); smyle11_time = smyle11_time.isel(L=slice(0,30))

In [ ]:
# smyle = xr.concat([smyle02,smyle05,smyle08,smyle11],'init')
smyle_time = xr.concat([smyle02_time,smyle05_time,smyle08_time,smyle11_time],'init')

In [ ]:
smyle_time['init'] = smyle_time['init']

In [ ]:
smyle = xr.concat([smyle02,smyle05,smyle08,smyle11],'init')

In [ ]:
obs = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/' + var + '.monthly.' + level + '.regrid.extremes.nc')['photoC_tot_zint_100m']

In [ ]:
del smyle02, smyle05, smyle08, smyle11

## mask land as NaNs

In [ ]:
ds = xr.open_dataset('/glade/work/smogen/SMYLE-extremes/OceanSODA-ETHZ_GRaCER_v2021a_1982-2020.nc')['temperature'].isel(time=0).drop('time')

ds = ds.where(ds.lat < 65)

In [ ]:
obs = obs.where(np.isnan(ds) == 0, np.NaN)

## Make a contingency table... confusing!

| Value      | Description |
| ----------- | ----------- |
| 1      | False negative       |
| 2      | True negative       |
| 3      | False positive       |
| 4      | True positive       |


In [ ]:
def sedi_index(data):
    hit_rate = (~np.isnan(data.where(data == 4))).sum(dim=('member','time'),skipna=True) / ((~np.isnan(data.where(data == 4))).sum(dim=('member','time'),skipna=True) + (~np.isnan(data.where(data == 1))).sum(dim=('member','time'),skipna=True))
    false_rate = (~np.isnan(data.where(data == 3))).sum(dim=('member','time'),skipna=True) / ((~np.isnan(data.where(data == 3))).sum(dim=('member','time'),skipna=True) + (~np.isnan(data.where(data == 2))).sum(dim=('member','time'),skipna=True))

    num = np.log10(false_rate) - np.log10(hit_rate) - (np.log10(1- false_rate)) + np.log10(1 - hit_rate)
    denom = np.log10(false_rate) + np.log10(hit_rate) + (np.log10(1- false_rate)) + np.log10(1 - hit_rate)

    sedi = (num) / (denom)
    return sedi

In [ ]:
def skill(mod_da,mod_time,obs_da):
    sedi_results = []
    forecast_acc_results = []
    bss_results = []
    for i in range(0,len(mod_da.L)): #len(mod_da.L)
        # print(i)
        # select a given member of the ensemble
        mod_da_L = mod_da.isel(L=i)
        results = []
        obs_range = []
        # put everything into a contingency table (key is listed above)
        for h in (mod_da.init):
        # h = '02'
            mod_da_L_I = mod_da_L.sel(init=h)
            for j in range(len(mod_da.M)):
                ens = mod_da_L_I.isel(M=j).rename({'Y':'time'})
            
                ens_time_year = mod_time.sel(init=h).isel(L=i).time.dt.year.data
                ens_time_month = mod_time.sel(init=h).isel(L=i).time.dt.month.data[0]
                ens_ts = ens.assign_coords(time=("time",ens_time_year))
                obs_ts = obs_da.where(obs_da.time.dt.month == ens_time_month,drop=True)
            
                obs_ts['time'] = obs_ts['time'].dt.year
            
                ens_ts, obs_ts = xr.align(ens_ts, obs_ts)

                ## randomize the values here!
                ens_ts_use = random.choices(ens_ts,k=len(ens_ts))
                ens_ts_use = xr.concat(ens_ts_use,'time')
                ens_ts_use['time'] = ens_ts['time']

                
                pos = (ens_ts_use.where(ens_ts_use.astype(int) == 1) + obs_ts + 2).rename('skill')
                neg = (ens_ts_use.where(ens_ts_use.astype(int) == 0) - obs_ts + 2).rename('skill')
                contingency = xr.merge([pos,neg])
                contingency = contingency.expand_dims('member')
                results.append(contingency)
                
                # isolate the month from observations 
                obs_ts.expand_dims('month')
                obs_ts['month'] = ens_time_month
                obs_range.append(obs_ts)       
        
        results_ds = xr.concat(results,'member')
        
        # run the SEDI score function (above), as in Jacox 2022
        sedi = sedi_index(results_ds.skill)
        sedi_results.append(sedi)
    
    ds = xr.concat(sedi_results,'L'); 
    return ds

In [ ]:
%%time
sedi_bootstrap = []
# for i in range(1000):
for k in range(1000):
    # print(k)
    sedi_score = skill(smyle,smyle_time,obs.drop('month'))
    sedi_score.expand_dims('bootstrap')
    sedi_score['bootstrap'] = k
    sedi_bootstrap.append(sedi_score)
    print(k)

In [ ]:
# 100 times, 10 members
tmp = xr.concat(sedi_bootstrap,'bootstrap')

In [ ]:
tmp.to_netcdf('./SEDI/MergedInits.FOSI.' + var + '.' + level + '.SEDI.BOOTSTRAP.nc')